In [ ]:
import os
import sys
import ast
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- Local Module Imports ---
SRC_FOLDER = r"C:\Users\ashto\ddi-prediction\src_test"
if SRC_FOLDER not in sys.path:
    sys.path.append(SRC_FOLDER)

from smpdb_protein_pathway import (
    PathwayMapper,
    get_targets_from_chembl_batch,
    UniprotConverter,
)

# =============================================================================
# CONFIGURATION & FILE PATHS
# =============================================================================
CHEMBL_ID_MAX_WORKERS = 8

# Grouped paths for easier local modifications
DATA_DIR = r"C:\Users\ashto\ddi-prediction\data"
DRUGBANK_XML_PATH = rf"{DATA_DIR}\raw\drugbank_full_database_V5.1.14.zip"
SMPDB_ZIP = rf"{DATA_DIR}\smpdb_pathways_data_csv\smpdb_proteins.csv.zip"

PER_DRUG_CSV = rf"{DATA_DIR}\per_drug_features.csv"
NEGATIVE_PAIRS_CSV = rf"{DATA_DIR}\negative_pairs.csv"
ADVERSE_PAIRS_CSV = rf"{DATA_DIR}\adverse_pairs.csv"

# Pair column mappings
NEG_COLS = ("drug1_id", "drug2_id")
ADV_COLS = ("drug1_id", "drug2_id")

FEATURE_COLS = [
    "smiles", "atc_codes_list", "target_uniprot_ids", 
    "target_fasta_sequences", "target_go_terms", 
    "target_pfam_domains", "target_pathway_neighbors"
]
LIST_COLS = FEATURE_COLS[1:] # All features except smiles are lists

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def safe_literal_eval(val):
    """Safely parse stringified lists from CSVs into actual Python lists."""
    if isinstance(val, list): return val
    if pd.isna(val): return []
    try:
        parsed = ast.literal_eval(str(val))
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

def has_elements(lst):
    """Helper to verify a list is valid and not empty."""
    return isinstance(lst, list) and len(lst) > 0

def chembl_molecule_id_by_name(drug_name):
    """Query ChEMBL API to find molecule ID by preferred name."""
    url = f"https://www.ebi.ac.uk/chembl/api/data/molecule.json?pref_name__iexact={drug_name}"
    try:
        res = requests.get(url, timeout=10).json()
        if res.get("molecules"):
            return res["molecules"][0]["molecule_chembl_id"]
    except Exception:
        pass
    return None

def update_completeness_flags(df):
    """Unified, dynamic function to compute feature completeness flags."""
    df["has_smiles"] = df["smiles"].notna() & (df["smiles"] != "")
    
    for col in LIST_COLS:
        flag_name = f"has_{col.replace('target_', '').replace('_list', '').replace('_sequences', '')}"
        df[flag_name] = df[col].apply(has_elements) if col in df.columns else False
        
    df["has_targets"] = df["target_uniprot_ids"].apply(has_elements)
    df["has_all_features"] = df["has_smiles"] & df["has_atc"] & df["hasHere is the streamlined version of your pipeline. The core logic remains entirely unchanged, but the execution is optimized for faster network calls, cleaner list handling, and vectorized pandas operations—which should make a noticeable difference, especially if you plan to run this on a larger node on the Palmetto cluster.

### The Streamlined Code

```python
import os
import sys
import ast
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- Local Module Imports ---
SRC_FOLDER = r"C:\Users\ashto\ddi-prediction\src_test"
if SRC_FOLDER not in sys.path:
    sys.path.append(SRC_FOLDER)

from smpdb_protein_pathway import (
    PathwayMapper,
    get_targets_from_chembl_batch,
    UniprotConverter,
    phi_infer_chembl_batch,
)

# =============================================================================
# CONFIGURATION & FILE PATHS
# =============================================================================
# Dynamically scale workers if running on a cluster, otherwise fallback to 8
MAX_WORKERS = min(32, (os.cpu_count() or 1) * 4) 
UNIPROT_FASTA_BASE = "[https://rest.uniprot.org/uniprotkb](https://rest.uniprot.org/uniprotkb)"

DRUGBANK_XML_PATH = r"C:\Users\ashto\ddi-prediction\data\raw\drugbank_full_database_V5.1.14.zip"
SMPDB_ZIP = r"C:\Users\ashto\ddi-prediction\data\smpdb_pathways_data_csv\smpdb_proteins.csv.zip"

PER_DRUG_CSV = r"C:\Users\ashto\ddi-prediction\data\per_drug_features.csv"
NEGATIVE_PAIRS_CSV = r"C:\Users\ashto\ddi-prediction\data\negative_pairs.csv"
ADVERSE_PAIRS_CSV = r"C:\Users\ashto\ddi-prediction\data\adverse_pairs.csv"

neg_d1, neg_d2 = "drug1_id", "drug2_id"
adv_d1, adv_d2 = "drug1_id", "drug2_id"
feature_cols = ["smiles", "atc_codes_list", "target_uniprot_ids", "target_fasta_sequences"]

# Initialize a global session to pool HTTP connections (massive speedup for APIs)
http_session = requests.Session()

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def parse_list_col(val):
    """Safely evaluates stringified lists from CSVs to actual Python lists."""
    if isinstance(val, str) and val.startswith('['):
        try: return ast.literal_eval(val)
        except ValueError: return []
    return val if isinstance(val, list) else []

def chembl_molecule_id_by_name(drug_name):
    url = f"[https://www.ebi.ac.uk/chembl/api/data/molecule.json?pref_name__iexact=](https://www.ebi.ac.uk/chembl/api/data/molecule.json?pref_name__iexact=){drug_name}"
    try:
        res = http_session.get(url, timeout=10).json()
        if res.get("molecules"):
            return res["molecules"][0]["molecule_chembl_id"]
    except Exception:
        pass
    return None

def attach_features(pair_df, col1, col2, features_df, feat_cols):
    """Vectorized merge for both drug1 and drug2 features."""
    d1_feats = features_df[feat_cols].add_suffix('_d1')
    d2_feats = features_df[feat_cols].add_suffix('_d2')
    return pair_df.merge(d1_feats, left_on=col1, right_index=True, how='left') \
                  .merge(d2_feats, left_on=col2, right_index=True, how='left')

def update_completeness_flags(df):
    """Vectorized calculation of feature completeness (much faster than .apply)."""
    df["has_smiles"] = df["smiles"].notna()
    df["has_atc"] = df["atc_codes_list"].str.len() > 0
    df["has_targets"] = (df["target_uniprot_ids"].str.len() > 0) | \
                        (df.get("target_other_ids", pd.Series(dtype=object)).str.len() > 0)
    
    for col, flag in [("target_fasta_sequences", "has_fasta"), 
                      ("target_go_terms", "has_go_terms"), 
                      ("target_pfam_domains", "has_pfam"), 
                      ("target_pathway_neighbors", "has_pathway_neighbors")]:
        if col in df.columns:
            df[flag] = df[col].str.len() > 0
            
    df["has_all_features"] = df["has_smiles"] & df["has_atc"] & df["has_targets"]
    return df

def uniprot_fetch_sequence(accession, timeout=10):
    if not accession: return None
    try:
        r = http_session.get(f"{UNIPROT_FASTA_BASE}/{accession}.fasta", timeout=timeout)
        if r.status_code == 200 and r.text:
            lines = r.text.strip().splitlines()
            return "".join(lines[1:]).strip() if lines and lines[0].startswith(">") else "".join(lines).strip()
    except Exception:
        pass
    return None

def _resolve_fasta_for_drug(dbid, uniprot_ids, existing_fasta):
    existing_set = set(existing_fasta)
    new_sequences = existing_set.copy()
    for acc in uniprot_ids:
        if seq := uniprot_fetch_sequence(acc):
            new_sequences.add(seq)
    return dbid, sorted(new_sequences)

# =============================================================================
# DATA INITIALIZATION
# =============================================================================
print("Loading initial datasets...")
per_drug_df = pd.read_csv(PER_DRUG_CSV)
negative_df = pd.read_csv(NEGATIVE_PAIRS_CSV)
adverse_df = pd.read_csv(ADVERSE_PAIRS_CSV)

# Force evaluation of stringified lists from CSV loading
list_columns = ["atc_codes_list", "target_uniprot_ids", "target_fasta_sequences", 
                "target_pathway_neighbors", "target_go_terms", "target_pfam_domains"]

for col in list_columns:
    if col in per_drug_df.columns:
        per_drug_df[col] = per_drug_df[col].apply(parse_list_col)
    else:
        per_drug_df[col] = [[] for _ in range(len(per_drug_df))]

per_drug_df = per_drug_df.set_index("drugbank_id", drop=False)

print("Initializing PathwayMapper (SMPDB + DrugBank)...")
pathway_mapper = PathwayMapper(xml_path=DRUGBANK_XML_PATH, smpdb_protein_zip=SMPDB_ZIP)

# =============================================================================
# STEP 1: Fast-track FASTA backfill for existing targets
# =============================================================================
mask = (per_drug_df["target_uniprot_ids"].str.len() > 0) & \
       (per_drug_df["target_fasta_sequences"].str.len() < per_drug_df["target_uniprot_ids"].str.len())
fasta_backfill_rows = per_drug_df[mask]

print(f"Resolving FASTA sequences for {len(fasta_backfill_rows)} drugs...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(_resolve_fasta_for_drug, row.drugbank_id, row.target_uniprot_ids, row.target_fasta_sequences): row.drugbank_id
        for row in fasta_backfill_rows.itertuples()
    }
    for i, future in enumerate(as_completed(futures), 1):
        dbid, seqs = future.result()
        if seqs:
            per_drug_df.at[dbid, "target_fasta_sequences"] = seqs
        if i % 100 == 0:
            print(f"  ... {i}/{len(fasta_backfill_rows)} FASTA queries processed")

# =============================================================================
# STEP 2: Pathway-based target inference (ChEMBL -> direct targets)
# =============================================================================
per_drug_df = update_completeness_flags(per_drug_df)
missing_targets_df = per_drug_df[~per_drug_df["has_targets"]]
print(f"\nResolving ChEMBL IDs for {len(missing_targets_df)} drugs missing targets...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    drug_to_chembl_id = {
        row.drugbank_id: chembl_id 
        for row, chembl_id in zip(
            missing_targets_df.itertuples(),
            executor.map(chembl_molecule_id_by_name, missing_targets_df["drug_name"])
        ) if chembl_id
    }

if drug_to_chembl_id:
    print(f"Querying ChEMBL for {len(drug_to_chembl_id)} molecules...")
    chembl_id_to_targets = get_targets_from_chembl_batch(list(drug_to_chembl_id.values()), min_pchembl=6.0)
    
    for dbid, chembl_id in drug_to_chembl_id.items():
        if targets := chembl_id_to_targets.get(chembl_id):
            existing = set(per_drug_df.at[dbid, "target_uniprot_ids"])
            per_drug_df.at[dbid, "target_uniprot_ids"] = sorted(existing | set(targets))

# =============================================================================
# STEP 3: Batch-fetch FASTA, GO, and Pfam for ALL missing accessions
# =============================================================================
# Fast flattening using list comprehension and set operations
all_missing_fasta_accs = {
    uid for row in per_drug_df.itertuples() if len(row.target_uniprot_ids) > len(row.target_fasta_sequences)
    for uid in row.target_uniprot_ids
}

if all_missing_fasta_accs:
    print(f"\nBatch-fetching sequences/GO/Pfam for {len(all_missing_fasta_accs)} unique UniProt accessions...")
    batch_converter = UniprotConverter(list(all_missing_fasta_accs))
    acc_to_seq = batch_converter.uniprot_to_sequence()
    acc_to_go = batch_converter.uniprot_to_GO_terms()
    acc_to_pfam = batch_converter.uniprot_to_pfams()

    for row in per_drug_df.itertuples():
        if not row.target_uniprot_ids: continue
        
        for data_dict, col in [(acc_to_seq, "target_fasta_sequences"), 
                               (acc_to_go, "target_go_terms"), 
                               (acc_to_pfam, "target_pfam_domains")]:
            existing_set = set(getattr(row, col))
            new_data = existing_set.copy()
            
            for acc in row.target_uniprot_ids:
                if val := data_dict.get(acc):
                    new_data.update([val] if isinstance(val, str) else val)
                    
            if new_data != existing_set:
                per_drug_df.at[row.drugbank_id, col] = sorted(new_data)

# =============================================================================
# STEP 4: Two-hop pathway neighbor expansion for ALL targets
# =============================================================================
all_current_targets = {uid for uids in per_drug_df["target_uniprot_ids"] for uid in uids}

if all_current_targets:
    print(f"\nExpanding {len(all_current_targets)} targets to two-hop neighbors...")
    protein_to_pathways = pathway_mapper.get_pathways_by_protein_batch(list(all_current_targets))
    
    # Fast unpacking of dictionary values into a flat set
    all_pathways = set().union(*protein_to_pathways.values())
    pathway_to_proteins = pathway_mapper.get_proteins_by_pathway_batch(list(all_pathways))

    for row in per_drug_df.itertuples():
        if not row.target_uniprot_ids: continue
        
        drug_pathways = set().union(*(protein_to_pathways.get(t, set()) for t in row.target_uniprot_ids))
        drug_neighbors = set().union(*(pathway_to_proteins.get(pw, set()) for pw in drug_pathways))
        
        if drug_neighbors:
            existing = set(per_drug_df.at[row.drugbank_id, "target_pathway_neighbors"])
            per_drug_df.at[row.drugbank_id, "target_pathway_neighbors"] = sorted(existing | drug_neighbors)

# =============================================================================
# STEP 5: Final Clean-up and Pair-Level Filtering
# =============================================================================
per_drug_df = per_drug_df.reset_index(drop=True)
per_drug_df = update_completeness_flags(per_drug_df)

print("\nFinal Feature Completeness Rates:")
print(per_drug_df[["has_smiles", "has_atc", "has_targets", "has_fasta", "has_go_terms", "has_pfam", "has_pathway_neighbors", "has_all_features"]].mean())

final_feature_cols = feature_cols + ["target_go_terms", "target_pfam_domains", "target_pathway_neighbors"]
complete_drug_ids = set(per_drug_df.loc[per_drug_df["has_all_features"], "drugbank_id"])

negative_final_df = negative_df[negative_df[neg_d1].isin(complete_drug_ids) & negative_df[neg_d2].isin(complete_drug_ids)].copy()
adverse_final_df  = adverse_df[adverse_df[adv_d1].isin(complete_drug_ids) & adverse_df[adv_d2].isin(complete_drug_ids)].copy()

per_drug_indexed = per_drug_df.set_index("drugbank_id")
negative_final_df = attach_features(negative_final_df, neg_d1, neg_d2, per_drug_indexed, final_feature_cols)
adverse_final_df = attach_features(adverse_final_df, adv_d1, adv_d2, per_drug_indexed, final_feature_cols)

print(f"\nFinal Pairs -> Negative: {len(negative_final_df)} | Adverse: {len(adverse_final_df)}")

Loading initial datasets...


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ashto\\ddi-prediction\\data\\per_drug_features.csv'